In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d
from scipy.signal import find_peaks
from statsmodels.tsa.seasonal import STL

try:
    import pywt
except ImportError:
    !pip install PyWavelets
    import pywt


BASE_OUTPUT_DIR = "/content/drive/MyDrive/Weather Trend Forecasting/processed_outputs"
OUTLIER_OUTPUT_DIR = os.path.join(BASE_OUTPUT_DIR, "outlier_processed_outputs")

DATA_PATH = os.path.join(
    OUTLIER_OUTPUT_DIR,
    "GlobalWeatherRepository_missing_outlier_cleaned.csv"
)

TREND_OUTPUT_DIR = os.path.join(
    OUTLIER_OUTPUT_DIR,
    "trend_analysis_outputs"
)

FIG_DIR = os.path.join(TREND_OUTPUT_DIR, "figures")
DATA_DIR = os.path.join(TREND_OUTPUT_DIR, "data")
REPORT_DIR = os.path.join(TREND_OUTPUT_DIR, "reports")

for p in [TREND_OUTPUT_DIR, FIG_DIR, DATA_DIR, REPORT_DIR]:
    os.makedirs(p, exist_ok=True)

df = pd.read_csv(DATA_PATH)
df["last_updated"] = pd.to_datetime(df["last_updated"], errors="coerce")
df = df.dropna(subset=["location_name", "last_updated"])

print(df.shape)
display(df.head())

Mounted at /content/drive
(141508, 41)


,last_updated,country,location_name,latitude,longitude,timezone,last_updated_epoch,temperature_celsius,temperature_fahrenheit,condition_text,...,air_quality_PM2.5,air_quality_PM10,air_quality_us-epa-index,air_quality_gb-defra-index,sunrise,sunset,moonrise,moonset,moon_phase,moon_illumination
0,2024-05-31 16:15:00,Belgium,'S Gravenjansdijk,51.25,3.63,Europe/Brussels,1717164900,16.0,60.8,Moderate rain,...,2.2,3.4,1,1,05:36 AM,09:52 PM,02:59 AM,02:09 PM,Waning Crescent,47
1,2024-06-01 16:30:00,Belgium,'S Gravenjansdijk,51.25,3.63,Europe/Brussels,1717252200,16.0,60.8,Overcast,...,6.5,19.6,1,1,05:35 AM,09:53 PM,03:12 AM,03:34 PM,Waning Crescent,35
2,2024-06-04 16:15:00,Belgium,'S Gravenjansdijk,51.25,3.63,Europe/Brussels,1717510500,18.0,64.4,Partly cloudy,...,5.7,6.0,1,1,05:33 AM,09:56 PM,03:56 AM,07:57 PM,Waning Crescent,8
3,2024-06-05 16:15:00,Belgium,'S Gravenjansdijk,51.25,3.63,Europe/Brussels,1717596900,15.0,59.0,Partly cloudy,...,1.2,2.4,1,1,05:33 AM,09:56 PM,04:17 AM,09:25 PM,Waning Crescent,3
4,2024-06-11 16:15:00,Belgium,'S Gravenjansdijk,51.25,3.63,Europe/Brussels,1718115300,15.2,59.4,Partly cloudy,...,0.5,0.9,1,1,05:30 AM,10:01 PM,10:16 AM,01:31 AM,Waxing Crescent,21


In [2]:
selected_cities = [
    "Tokyo",
    "Baghdad",
    "Bern",
    "Suva",
    "Dakar",
    "Kyiv",
    "Accra",
    "Kabul",
    "Valletta",
    "Warsaw"
]

selected_features = [
    "temperature_celsius",
    "humidity",
    "pressure_mb",
    "precip_mm",
    "wind_kph",
    "air_quality_PM2.5",
    "air_quality_PM10",
    "air_quality_Carbon_Monoxide",
    "visibility_km",
    "uv_index"
]

selected_features = [f for f in selected_features if f in df.columns]
print("Cities:", selected_cities)
print("Features:", selected_features)

Cities: ['Tokyo', 'Baghdad', 'Bern', 'Suva', 'Dakar', 'Kyiv', 'Accra', 'Kabul', 'Valletta', 'Warsaw']
Features: ['temperature_celsius', 'humidity', 'pressure_mb', 'precip_mm', 'wind_kph', 'air_quality_PM2.5', 'air_quality_PM10', 'air_quality_Carbon_Monoxide', 'visibility_km', 'uv_index']


In [3]:
import re
import unicodedata

def safe_name(x):
    # Replace all non-alphanumeric chars with "_"
    x = str(x)
    x = unicodedata.normalize("NFKD", x)
    x = x.encode("ascii", "ignore").decode("ascii")
    x = re.sub(r"[^A-Za-z0-9]+", "_", x)
    x = re.sub(r"_+", "_", x)
    x = x.strip("_")
    if x == "":
        x = "unknown"
    return x

def infer_regular_frequency(city_df):
    diffs = city_df["last_updated"].sort_values().diff().dropna()
    if len(diffs) == 0:
        return "1D"
    median_diff = diffs.median()
    hours = median_diff.total_seconds() / 3600
    if hours <= 2:
        return "1H"
    elif hours <= 8:
        return "6H"
    elif hours <= 18:
        return "12H"
    else:
        return "1D"

def prepare_city_feature_series(df, city, feature):
    g = df[df["location_name"] == city].copy()
    g = g.sort_values("last_updated")
    g[feature] = pd.to_numeric(g[feature], errors="coerce")
    freq = infer_regular_frequency(g)
    ts = (
        g[["last_updated", feature]]
        .dropna()
        .drop_duplicates(subset=["last_updated"])
        .set_index("last_updated")
        .sort_index()
    )
    ts_regular = ts.resample(freq).mean()
    ts_regular[feature] = ts_regular[feature].interpolate(
        method="time",
        limit_direction="both"
    )
    return ts_regular, freq

def freq_to_hours(freq):
    if freq.endswith("H"):
        return float(freq.replace("H", ""))
    if freq.endswith("D"):
        return float(freq.replace("D", "")) * 24
    return 24.0

def gaussian_sigma_by_window(window_points):
    # approximate: 6 sigma covers the window
    return max(window_points / 6, 1)

def get_smoothing_sigmas(freq):
    step_hours = freq_to_hours(freq)
    week_points = max(int((7 * 24) / step_hours), 3)
    month_points = max(int((30 * 24) / step_hours), 3)
    year_points = max(int((365 * 24) / step_hours), 3)
    return {
        "weekly": gaussian_sigma_by_window(week_points),
        "monthly": gaussian_sigma_by_window(month_points),
        "yearly": gaussian_sigma_by_window(year_points),
        "weekly_window_points": week_points,
        "monthly_window_points": month_points,
        "yearly_window_points": year_points
    }

def choose_stl_period(freq, n):
    step_hours = freq_to_hours(freq)
    # prefer monthly-ish seasonality if enough length;
    # otherwise weekly; otherwise small fallback.
    daily = max(int(24 / step_hours), 2)
    weekly = max(int(7 * 24 / step_hours), 2)
    monthly = max(int(30 * 24 / step_hours), 2)
    if n >= monthly * 3:
        return monthly
    elif n >= weekly * 3:
        return weekly
    else:
        return min(max(daily, 2), max(n // 3, 2))

def save_series_csv(series_df, path):
    out = series_df.copy()
    out = out.reset_index()
    out.to_csv(path, index=False)

In [4]:
# Analysis function for one city-feature pair

summary_records = []

def run_trend_analysis_one(city, feature):
    city_safe = safe_name(city)
    feature_safe = safe_name(feature)
    pair_dir = os.path.join(DATA_DIR, city_safe, feature_safe)
    pair_fig_dir = os.path.join(FIG_DIR, city_safe, feature_safe)
    os.makedirs(pair_dir, exist_ok=True)
    os.makedirs(pair_fig_dir, exist_ok=True)
    ts, freq = prepare_city_feature_series(df, city, feature)
    if len(ts) < 30:
        return
    y = ts[feature].values.astype(float)
    t = ts.index
    start_time = t.min()
    end_time = t.max()
    duration = end_time - start_time
    duration_days = duration.total_seconds() / 86400

    # Raw + Gaussian smoothing
    sigmas = get_smoothing_sigmas(freq)
    y_week = gaussian_filter1d(y, sigma=sigmas["weekly"])
    y_month = gaussian_filter1d(y, sigma=sigmas["monthly"])
    y_year = gaussian_filter1d(y, sigma=sigmas["yearly"])
    smooth_df = pd.DataFrame({
        "last_updated": t,
        "raw": y,
        "weekly_gaussian": y_week,
        "monthly_gaussian": y_month,
        "yearly_gaussian": y_year
    })
    smooth_df.to_csv(
        os.path.join(pair_dir, f"{city_safe}_{feature_safe}_gaussian_smoothing.csv"),
        index=False
    )
    fig, axes = plt.subplots(1, 4, figsize=(24, 5))
    axes[0].plot(t, y, linewidth=1)
    axes[0].set_title("Raw Signal")
    axes[0].set_xlabel("Time")
    axes[0].set_ylabel(feature)

    axes[1].plot(t, y, linewidth=0.6, alpha=0.4)
    axes[1].plot(t, y_week, linewidth=2)
    axes[1].set_title("Weekly Gaussian Trend")
    axes[1].set_xlabel("Time")

    axes[2].plot(t, y, linewidth=0.6, alpha=0.4)
    axes[2].plot(t, y_month, linewidth=2)
    axes[2].set_title("Monthly Gaussian Trend")
    axes[2].set_xlabel("Time")

    axes[3].plot(t, y, linewidth=0.6, alpha=0.4)
    axes[3].plot(t, y_year, linewidth=2)
    axes[3].set_title("Yearly Gaussian Trend")
    axes[3].set_xlabel("Time")

    fig.suptitle(
        f"{city} - {feature} | {start_time} to {end_time} | {duration_days:.1f} days",
        fontsize=14
    )
    plt.tight_layout()
    plt.savefig(
        os.path.join(pair_fig_dir, f"{city_safe}_{feature_safe}_01_raw_gaussian.png"),
        dpi=300
    )
    plt.close()

    # STL decomposition
    stl_period = choose_stl_period(freq, len(y))
    stl = STL(
        y,
        period=stl_period,
        robust=True
    )
    result = stl.fit()
    stl_df = pd.DataFrame({
        "last_updated": t,
        "observed": y,
        "trend": result.trend,
        "seasonal": result.seasonal,
        "residual": result.resid
    })
    stl_df.to_csv(
        os.path.join(pair_dir, f"{city_safe}_{feature_safe}_stl_decomposition.csv"),
        index=False
    )

    fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)

    axes[0].plot(t, y, linewidth=1)
    axes[0].set_title("Observed")

    axes[1].plot(t, result.trend, linewidth=1.5)
    axes[1].set_title("STL Trend")

    axes[2].plot(t, result.seasonal, linewidth=1)
    axes[2].set_title("STL Seasonal Component")

    axes[3].plot(t, result.resid, linewidth=1)
    axes[3].axhline(0, linewidth=1)
    axes[3].set_title("STL Residual")

    fig.suptitle(f"STL Decomposition | {city} - {feature} | period={stl_period}", fontsize=14)
    plt.tight_layout()
    plt.savefig(
        os.path.join(pair_fig_dir, f"{city_safe}_{feature_safe}_02_stl_decomposition.png"),
        dpi=300
    )
    plt.close()

    # Fourier analysis + top 3 frequency filtering
    y_detrended = y - np.nanmean(y)
    n = len(y_detrended)
    step_hours = freq_to_hours(freq)
    fft_vals = np.fft.rfft(y_detrended)
    freqs = np.fft.rfftfreq(n, d=step_hours)
    power = np.abs(fft_vals) ** 2
    valid = freqs > 0
    valid_freqs = freqs[valid]
    valid_power = power[valid]
    if len(valid_freqs) >= 3:
        peaks, _ = find_peaks(valid_power)
        if len(peaks) >= 3:
            top_peak_indices = peaks[np.argsort(valid_power[peaks])[-3:]][::-1]
        else:
            top_peak_indices = np.argsort(valid_power)[-3:][::-1]

        top_freqs = valid_freqs[top_peak_indices]
    else:
        top_freqs = []
    spectrum_df = pd.DataFrame({
        "frequency_cycles_per_hour": valid_freqs,
        "period_hours": 1 / valid_freqs,
        "power": valid_power
    })
    spectrum_df.to_csv(
        os.path.join(pair_dir, f"{city_safe}_{feature_safe}_fft_spectrum.csv"),
        index=False
    )

    plt.figure(figsize=(12, 5))
    plt.plot(1 / valid_freqs, valid_power, linewidth=1)
    plt.xscale("log")
    plt.xlabel("Period (hours, log scale)")
    plt.ylabel("Power")
    plt.title(f"FFT Spectrum | {city} - {feature}")
    plt.tight_layout()
    plt.savefig(
        os.path.join(pair_fig_dir, f"{city_safe}_{feature_safe}_03_fft_spectrum.png"),
        dpi=300
    )
    plt.close()
    filtered_components = {}

    for k, f0 in enumerate(top_freqs, start=1):
        band_width = f0 * 0.15
        mask = np.abs(np.fft.rfftfreq(n, d=step_hours) - f0) <= band_width
        fft_filtered = np.zeros_like(fft_vals)
        fft_filtered[mask] = fft_vals[mask]
        y_filtered = np.fft.irfft(fft_filtered, n=n)
        filtered_components[f"component_{k}_freq"] = f0
        filtered_components[f"component_{k}_period_hours"] = 1 / f0
        plt.figure(figsize=(12, 5))
        plt.plot(t, y, linewidth=0.6, alpha=0.4, label="Original")
        plt.plot(t, y_filtered + np.nanmean(y), linewidth=2, label=f"Filtered component {k}")
        plt.title(
            f"FFT Filtered Component {k} | {city} - {feature}\n"
            f"Period ≈ {1 / f0:.2f} hours"
        )
        plt.xlabel("Time")
        plt.ylabel(feature)
        plt.legend()
        plt.tight_layout()
        plt.savefig(
            os.path.join(pair_fig_dir, f"{city_safe}_{feature_safe}_04_fft_filtered_component_{k}.png"),
            dpi=300
        )
        plt.close()

    # Morlet CWT time-frequency heatmap
    max_scale = min(256, max(32, len(y) // 2))
    scales = np.arange(1, max_scale)
    coefficients, frequencies = pywt.cwt(
        y_detrended,
        scales,
        "morl",
        sampling_period=step_hours
    )
    power_cwt = np.abs(coefficients) ** 2
    periods = 1 / frequencies

    cwt_power_df = pd.DataFrame(
        power_cwt,
        columns=[str(x) for x in t],
        index=periods
    )
    cwt_power_df.index.name = "period_hours"
    cwt_power_df.to_csv(
        os.path.join(pair_dir, f"{city_safe}_{feature_safe}_cwt_morlet_power.csv")
    )
    plt.figure(figsize=(14, 6))
    plt.imshow(
        power_cwt,
        aspect="auto",
        origin="lower",
        extent=[0, len(t), periods.min(), periods.max()]
    )
    plt.yscale("log")
    plt.colorbar(label="Wavelet Power")
    plt.xlabel("Time index")
    plt.ylabel("Period (hours, log scale)")
    plt.title(f"Morlet CWT Time-Frequency Heatmap | {city} - {feature}")
    plt.tight_layout()
    plt.savefig(
        os.path.join(pair_fig_dir, f"{city_safe}_{feature_safe}_05_morlet_cwt_heatmap.png"),
        dpi=300
    )
    plt.close()

    summary_records.append({
        "city": city,
        "feature": feature,
        "record_count_after_resampling": len(ts),
        "frequency": freq,
        "start_time": start_time,
        "end_time": end_time,
        "duration_days": duration_days,
        "weekly_gaussian_sigma": sigmas["weekly"],
        "monthly_gaussian_sigma": sigmas["monthly"],
        "yearly_gaussian_sigma": sigmas["yearly"],
        "weekly_window_points": sigmas["weekly_window_points"],
        "monthly_window_points": sigmas["monthly_window_points"],
        "yearly_window_points": sigmas["yearly_window_points"],
        "stl_period": stl_period,
        **filtered_components
    })

In [5]:
# Run 10 × 10 = 100 trend analyses

for city in selected_cities:
    for feature in selected_features:
        print(f"Running trend analysis: {city} | {feature}")
        try:
            run_trend_analysis_one(city, feature)
        except Exception as e:
            summary_records.append({
                "city": city,
                "feature": feature,
                "error": str(e)
            })
            print(f"Failed: {city} | {feature} | {e}")

trend_summary = pd.DataFrame(summary_records)
summary_path = os.path.join(
    REPORT_DIR,
    "trend_analysis_summary_10cities_10features.csv"
)

trend_summary.to_csv(summary_path, index=False)
display(trend_summary.head(20))

print("Trend analysis completed.")
print("Summary saved to:")
print(summary_path)
print("Figures saved to:")
print(FIG_DIR)
print("Numerical results saved to:")
print(DATA_DIR)

Running trend analysis: Tokyo | temperature_celsius
Running trend analysis: Tokyo | humidity
Running trend analysis: Tokyo | pressure_mb
Running trend analysis: Tokyo | precip_mm
Running trend analysis: Tokyo | wind_kph
Running trend analysis: Tokyo | air_quality_PM2.5
Running trend analysis: Tokyo | air_quality_PM10
Running trend analysis: Tokyo | air_quality_Carbon_Monoxide
Running trend analysis: Tokyo | visibility_km
Running trend analysis: Tokyo | uv_index
Running trend analysis: Baghdad | temperature_celsius
Running trend analysis: Baghdad | humidity
Running trend analysis: Baghdad | pressure_mb
Running trend analysis: Baghdad | precip_mm
Running trend analysis: Baghdad | wind_kph
Running trend analysis: Baghdad | air_quality_PM2.5
Running trend analysis: Baghdad | air_quality_PM10
Running trend analysis: Baghdad | air_quality_Carbon_Monoxide
Running trend analysis: Baghdad | visibility_km
Running trend analysis: Baghdad | uv_index
Running trend analysis: Bern | temperature_celsi

,city,feature,record_count_after_resampling,frequency,start_time,end_time,duration_days,weekly_gaussian_sigma,monthly_gaussian_sigma,yearly_gaussian_sigma,weekly_window_points,monthly_window_points,yearly_window_points,stl_period,component_1_freq,component_1_period_hours,component_2_freq,component_2_period_hours,component_3_freq,component_3_period_hours
0,Tokyo,temperature_celsius,730,1D,2024-05-16,2026-05-15,729.0,1.166667,5.0,60.833333,7,30,365,30,0.000114,8760.000000,0.000228,4380.000000,0.000342,2920.000000
1,Tokyo,humidity,730,1D,2024-05-16,2026-05-15,729.0,1.166667,5.0,60.833333,7,30,365,30,0.000114,8760.000000,0.000342,2920.000000,0.000856,1168.000000
2,Tokyo,pressure_mb,730,1D,2024-05-16,2026-05-15,729.0,1.166667,5.0,60.833333,7,30,365,30,0.000114,8760.000000,0.000228,4380.000000,0.000342,2920.000000
3,Tokyo,precip_mm,730,1D,2024-05-16,2026-05-15,729.0,1.166667,5.0,60.833333,7,30,365,30,0.019806,50.489914,0.020491,48.802228,0.015868,63.021583
4,Tokyo,wind_kph,730,1D,2024-05-16,2026-05-15,729.0,1.166667,5.0,60.833333,7,30,365,30,0.000114,8760.000000,0.000228,4380.000000,0.005023,199.090909
5,Tokyo,air_quality_PM2.5,730,1D,2024-05-16,2026-05-15,729.0,1.166667,5.0,60.833333,7,30,365,30,0.000514,1946.666667,0.002511,398.181818,0.000228,4380.000000
6,Tokyo,air_quality_PM10,730,1D,2024-05-16,2026-05-15,729.0,1.166667,5.0,60.833333,7,30,365,30,0.000514,1946.666667,0.003824,261.492537,0.002511,398.181818
7,Tokyo,air_quality_Carbon_Monoxide,730,1D,2024-05-16,2026-05-15,729.0,1.166667,5.0,60.833333,7,30,365,30,0.000228,4380.000000,0.000342,2920.000000,0.003425,292.000000
8,Tokyo,visibility_km,730,1D,2024-05-16,2026-05-15,729.0,1.166667,5.0,60.833333,7,30,365,30,0.000285,3504.000000,0.001256,796.363636,0.005651,176.969697
9,Tokyo,uv_index,730,1D,2024-05-16,2026-05-15,729.0,1.166667,5.0,60.833333,7,30,365,30,0.000285,3504.000000,0.000571,1752.000000,0.005936,168.461538


Trend analysis completed.
Summary saved to:
/content/drive/MyDrive/Weather Trend Forecasting/processed_outputs/outlier_processed_outputs/trend_analysis_outputs/reports/trend_analysis_summary_10cities_10features.csv
Figures saved to:
/content/drive/MyDrive/Weather Trend Forecasting/processed_outputs/outlier_processed_outputs/trend_analysis_outputs/figures
Numerical results saved to:
/content/drive/MyDrive/Weather Trend Forecasting/processed_outputs/outlier_processed_outputs/trend_analysis_outputs/data


In [6]:
# Trend Metrics Extraction
import os
import numpy as np
import pandas as pd
from scipy.stats import kendalltau
from scipy.signal import find_peaks

BASE_OUTPUT_DIR = "/content/drive/MyDrive/Weather Trend Forecasting/processed_outputs"
OUTLIER_OUTPUT_DIR = os.path.join(
    BASE_OUTPUT_DIR,
    "outlier_processed_outputs"
)

TREND_OUTPUT_DIR = os.path.join(
    OUTLIER_OUTPUT_DIR,
    "trend_analysis_outputs"
)

DATA_DIR = os.path.join(
    TREND_OUTPUT_DIR,
    "data"
)

REPORT_DIR = os.path.join(
    TREND_OUTPUT_DIR,
    "trend_metrics_reports"
)

os.makedirs(REPORT_DIR, exist_ok=True)
summary_file = os.path.join(
    TREND_OUTPUT_DIR,
    "reports",
    "trend_analysis_summary_10cities_10features.csv"
)

summary_df = pd.read_csv(summary_file)
metrics_records = []

In [7]:
# Helper functions
def mann_kendall_test(y):
    x = np.arange(len(y))
    tau, p = kendalltau(x, y)
    if p < 0.001:
        sig = "***"
    elif p < 0.01:
        sig = "**"
    elif p < 0.05:
        sig = "*"
    else:
        sig = "ns"
    return tau, p, sig


def compute_strength(component, residual):
    num = np.var(residual)
    den = np.var(component + residual)
    if den <= 1e-12:
        return 0
    return max(0, 1 - num / den)


def compute_seasonal_amplitude_variability(
        seasonal,
        window=30):
    s = pd.Series(seasonal)
    rolling_amp = (s.rolling(window,center=True
        ).max()-s.rolling(window,center=True).min())

    return np.nanstd(rolling_amp)

In [8]:
# Process all 100 trend groups
for _, row in summary_df.iterrows():
    city = row["city"]
    feature = row["feature"]
    city_safe = str(city).replace(" ", "_")
    feature_safe = str(feature).replace("/", "_")
    pair_dir = os.path.join(
        DATA_DIR,
        city_safe,
        feature_safe)
    stl_file = os.path.join(
        pair_dir,
        f"{city_safe}_{feature_safe}_stl_decomposition.csv")
    fft_file = os.path.join(
        pair_dir,
        f"{city_safe}_{feature_safe}_fft_spectrum.csv")
    if ( not os.path.exists(stl_file)
        or
        not os.path.exists(fft_file)):
        continue
    try:
        stl_df = pd.read_csv(stl_file)
        fft_df = pd.read_csv(fft_file)

        # Trend slope
        trend = stl_df["trend"].values
        x = np.arange(len(trend))
        slope = np.polyfit(x,trend,1)[0]
        trend_change = (trend[-1]-trend[0])

        # Mann-Kendall
        tau, p_value, significance = (
            mann_kendall_test(trend))

        # Dominant period
        fft_df = fft_df.sort_values(
            "power",
            ascending=False)
        dom_hours = (fft_df.iloc[0]["period_hours"])
        dom_days = (dom_hours / 24)

        # Seasonal / trend strength
        seasonal = (stl_df["seasonal"].values)
        residual = (stl_df["residual"].values)
        trend_strength = (compute_strength(trend,residual))
        seasonal_strength = (compute_strength(seasonal,residual))

        # Seasonal amplitude variability
        seasonal_variability = (compute_seasonal_amplitude_variability(seasonal))
        seasonal_std = (np.std(seasonal))

        seasonal_range = (np.max(seasonal)-np.min(seasonal))

        metrics_records.append({
            "city":city,
            "feature":feature,
            "trend_slope":slope,
            "trend_total_change":trend_change,
            "mann_kendall_tau":tau,
            "mann_kendall_p":p_value,
            "mann_kendall_significance":significance,
            "dominant_period_hours":dom_hours,
            "dominant_period_days":dom_days,
            "seasonal_strength":seasonal_strength,
            "trend_strength":trend_strength,
            "seasonal_variability":seasonal_variability,
            "seasonal_std":seasonal_std,
            "seasonal_range":seasonal_range})

    except Exception as e:
        print(city,feature,e)

In [9]:
# Save
metrics_df = pd.DataFrame(metrics_records)
metrics_df = metrics_df.sort_values(["city","feature"])
display(metrics_df.head(20))
save_path = os.path.join(
    REPORT_DIR,
    "trend_quantitative_metrics.csv")
metrics_df.to_csv(save_path,index=False)
print("Saved:")
print(save_path)

,city,feature,trend_slope,trend_total_change,mann_kendall_tau,mann_kendall_p,mann_kendall_significance,dominant_period_hours,dominant_period_days,seasonal_strength,trend_strength,seasonal_variability,seasonal_std,seasonal_range
60,Accra,air_quality_Carbon_Monoxide,-0.354308,-86.056387,-0.570017,1.603448e-117,***,17520.0,730.0,0.176146,0.839472,47.407349,25.533845,273.283850
59,Accra,air_quality_PM10,-0.020799,5.463837,-0.039330,1.118065e-01,ns,8760.0,365.0,0.000000,0.581547,21.840661,7.676383,90.876697
55,Accra,humidity,0.032939,21.238073,0.614431,3.157581e-136,***,17520.0,730.0,0.055736,0.659279,2.790222,2.665484,27.004133
57,Accra,precip_mm,-0.000036,-0.033696,-0.215386,3.086379e-18,***,4380.0,182.5,0.024697,0.022774,0.147360,0.042343,0.566007
56,Accra,pressure_mb,-0.004050,-1.276302,-0.371622,5.037714e-51,***,8760.0,365.0,0.284810,0.816972,0.584684,0.654477,4.608190
54,Accra,temperature_celsius,-0.004482,-4.461463,-0.305929,3.848431e-35,***,17520.0,730.0,0.160133,0.740671,0.859278,0.628130,5.804495
62,Accra,uv_index,-0.010775,-6.749326,-0.870726,1.727318e-271,***,17520.0,730.0,0.109167,0.884917,0.777982,0.373162,3.280944
61,Accra,visibility_km,-0.000553,-1.126054,-0.179217,4.295817e-13,***,8760.0,365.0,0.082002,0.351728,1.470229,0.934084,6.780781
58,Accra,wind_kph,-0.008096,-5.534206,-0.214056,4.953047e-18,***,8760.0,365.0,0.207985,0.624934,3.507391,2.035601,21.825837
15,Baghdad,air_quality_Carbon_Monoxide,-0.451784,241.588658,-0.180382,3.031233e-13,***,17520.0,730.0,0.029634,0.341952,1279.852386,483.381159,5245.500432


Saved:
/content/drive/MyDrive/Weather Trend Forecasting/processed_outputs/outlier_processed_outputs/trend_analysis_outputs/trend_metrics_reports/trend_quantitative_metrics.csv


In [10]:
# Monthly Seasonal Amplitude Variability from STL Seasonal Component

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BASE_OUTPUT_DIR = "/content/drive/MyDrive/Weather Trend Forecasting/processed_outputs"
OUTLIER_OUTPUT_DIR = os.path.join(BASE_OUTPUT_DIR, "outlier_processed_outputs")
TREND_OUTPUT_DIR = os.path.join(OUTLIER_OUTPUT_DIR, "trend_analysis_outputs")

DATA_DIR = os.path.join(TREND_OUTPUT_DIR, "data")
REPORT_DIR = os.path.join(TREND_OUTPUT_DIR, "trend_metrics_reports")
FIG_DIR = os.path.join(TREND_OUTPUT_DIR, "figures", "monthly_seasonal_amplitude")

os.makedirs(REPORT_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

summary_file = os.path.join(
    TREND_OUTPUT_DIR,
    "reports",
    "trend_analysis_summary_10cities_10features.csv")
summary_df = pd.read_csv(summary_file)

def safe_name(x):
    return str(x).replace(" ", "_").replace("/", "_").replace("\\", "_").replace(".", "_")

monthly_records = []
monthly_summary_records = []

for _, row in summary_df.iterrows():
    city = row["city"]
    feature = row["feature"]
    city_safe = safe_name(city)
    feature_safe = safe_name(feature)
    stl_file = os.path.join(
        DATA_DIR,
        city_safe,
        feature_safe,
        f"{city_safe}_{feature_safe}_stl_decomposition.csv"
    )
    if not os.path.exists(stl_file):
        continue
    stl_df = pd.read_csv(stl_file)
    stl_df["last_updated"] = pd.to_datetime(stl_df["last_updated"], errors="coerce")
    stl_df = stl_df.dropna(subset=["last_updated", "seasonal"])

    # Natural month grouping
    stl_df["year_month"] = stl_df["last_updated"].dt.to_period("M").astype(str)
    monthly_amp_df = (
        stl_df.groupby("year_month")["seasonal"]
        .agg(
            monthly_seasonal_max="max",
            monthly_seasonal_min="min",
            monthly_seasonal_mean="mean",
            monthly_seasonal_std="std",
            point_count="count")
        .reset_index())

    monthly_amp_df["monthly_seasonal_amplitude"] = (
        monthly_amp_df["monthly_seasonal_max"]
        - monthly_amp_df["monthly_seasonal_min"])

    monthly_amp_df["city"] = city
    monthly_amp_df["feature"] = feature
    monthly_records.append(monthly_amp_df)
    amps = monthly_amp_df["monthly_seasonal_amplitude"].dropna()

    if len(amps) > 0:
        amp_mean = amps.mean()
        amp_std = amps.std()
        amp_var = amps.var()
        amp_range = amps.max() - amps.min()
        amp_cv = amp_std / amp_mean if amp_mean != 0 else np.nan
    else:
        amp_mean = np.nan
        amp_std = np.nan
        amp_var = np.nan
        amp_range = np.nan
        amp_cv = np.nan

    monthly_summary_records.append({
        "city": city,
        "feature": feature,
        "num_months": len(amps),
        "monthly_seasonal_amplitude_mean": amp_mean,
        "monthly_seasonal_amplitude_std": amp_std,
        "monthly_seasonal_amplitude_variance": amp_var,
        "monthly_seasonal_amplitude_range": amp_range,
        "monthly_seasonal_amplitude_cv": amp_cv
    })

    # Save per-pair monthly amplitude table
    pair_out_dir = os.path.join(DATA_DIR, city_safe, feature_safe)
    monthly_amp_df.to_csv(
        os.path.join(pair_out_dir, f"{city_safe}_{feature_safe}_monthly_seasonal_amplitude.csv"),
        index=False)

    # Plot monthly amplitude
    plt.figure(figsize=(12, 5))
    plt.plot(
        monthly_amp_df["year_month"],
        monthly_amp_df["monthly_seasonal_amplitude"],
        marker="o",
        linewidth=2)
    plt.xticks(rotation=45, ha="right")
    plt.title(f"Monthly STL Seasonal Amplitude | {city} - {feature}")
    plt.xlabel("Month")
    plt.ylabel("Monthly seasonal amplitude: max(seasonal) - min(seasonal)")
    plt.tight_layout()

    plt.savefig(
        os.path.join(FIG_DIR, f"{city_safe}_{feature_safe}_monthly_seasonal_amplitude.png"),
        dpi=300)
    plt.close()


# Save all monthly-level results
all_monthly_amp_df = pd.concat(monthly_records, ignore_index=True)
all_monthly_amp_path = os.path.join(
    REPORT_DIR,
    "monthly_seasonal_amplitude_all_groups.csv")
all_monthly_amp_df.to_csv(all_monthly_amp_path, index=False)

# Save summary results
monthly_summary_df = pd.DataFrame(monthly_summary_records)
monthly_summary_path = os.path.join(
    REPORT_DIR,
    "monthly_seasonal_amplitude_summary.csv")

monthly_summary_df.to_csv(monthly_summary_path, index=False)
display(monthly_summary_df.head(20))

print("Saved monthly amplitude details to:")
print(all_monthly_amp_path)
print("Saved monthly amplitude summary to:")
print(monthly_summary_path)
print("Saved monthly amplitude figures to:")
print(FIG_DIR)

,city,feature,num_months,monthly_seasonal_amplitude_mean,monthly_seasonal_amplitude_std,monthly_seasonal_amplitude_variance,monthly_seasonal_amplitude_range,monthly_seasonal_amplitude_cv
0,Tokyo,temperature_celsius,25,6.601140,1.928211,3.717998e+00,8.362126,0.292103
1,Tokyo,humidity,25,38.979309,14.045239,1.972687e+02,52.991201,0.360326
2,Tokyo,pressure_mb,25,14.185969,3.734858,1.394917e+01,14.907449,0.263278
3,Tokyo,precip_mm,25,0.192496,0.153725,2.363138e-02,0.454690,0.798588
4,Tokyo,wind_kph,25,20.316968,6.814359,4.643549e+01,32.869954,0.335402
5,Tokyo,air_quality_PM2.5,25,49.674647,8.538017,7.289773e+01,31.594040,0.171879
6,Tokyo,air_quality_PM10,25,53.804762,11.441691,1.309123e+02,40.707914,0.212652
7,Tokyo,air_quality_Carbon_Monoxide,25,422.558944,191.975331,3.685453e+04,759.453479,0.454316
8,Tokyo,visibility_km,25,1.645650,2.274721,5.174356e+00,5.350496,1.382263
9,Tokyo,uv_index,25,0.547851,0.744636,5.544834e-01,2.825368,1.359196


Saved monthly amplitude details to:
/content/drive/MyDrive/Weather Trend Forecasting/processed_outputs/outlier_processed_outputs/trend_analysis_outputs/trend_metrics_reports/monthly_seasonal_amplitude_all_groups.csv
Saved monthly amplitude summary to:
/content/drive/MyDrive/Weather Trend Forecasting/processed_outputs/outlier_processed_outputs/trend_analysis_outputs/trend_metrics_reports/monthly_seasonal_amplitude_summary.csv
Saved monthly amplitude figures to:
/content/drive/MyDrive/Weather Trend Forecasting/processed_outputs/outlier_processed_outputs/trend_analysis_outputs/figures/monthly_seasonal_amplitude
